# Clase 094 — MDS, Isomap, t-SNE, (UMAP) y LDA

Técnicas de reducción de dimensionalidad más allá de PCA. Cada una preserva algo distinto: distancias (MDS), geodésicas (Isomap), vecindarios locales (t-SNE), separación entre clases (LDA supervisado).

> **Nota:** UMAP se menciona por completitud pero **no está instalado** en este entorno; usamos `MDS`, `Isomap`, `TSNE` y `LDA` de scikit-learn.

Requiere: `numpy`, `scikit-learn`, `matplotlib`. t-SNE es lento: submuestreamos a 500 puntos.

## 1. Dataset: swiss roll y digits

Swiss roll para métodos de manifold; `digits` (subset de 500) para t-SNE/LDA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll, load_digits
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
rng = np.random.default_rng(42)

Xsr, color = make_swiss_roll(n_samples=800, noise=0.05, random_state=42)

Xd_full, yd_full = load_digits(return_X_y=True)
idx = rng.choice(len(Xd_full), size=500, replace=False)
Xd, yd = Xd_full[idx], yd_full[idx]
Xd_s = StandardScaler().fit_transform(Xd)
print("swiss roll:", Xsr.shape, "| digits subset:", Xd.shape)

## 2. PCA vs MDS vs Isomap sobre swiss roll

Isomap usa distancias **geodésicas** (camino sobre el grafo de vecinos) y desenrolla; PCA/MDS aplanan.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, Isomap

pca = PCA(n_components=2, random_state=42).fit_transform(Xsr)
mds = MDS(n_components=2, n_init=1, max_iter=100, random_state=42,
          normalized_stress="auto").fit_transform(Xsr)
iso = Isomap(n_neighbors=10, n_components=2).fit_transform(Xsr)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, Z, name in zip(ax, (pca, mds, iso), ("PCA", "MDS", "Isomap")):
    a.scatter(Z[:, 0], Z[:, 1], c=color, cmap="Spectral", s=8)
    a.set_title(name); a.set_xticks([]); a.set_yticks([])
plt.suptitle("Solo Isomap 'desenrolla' el swiss roll")
plt.tight_layout(); plt.show()
assert iso.shape == (800, 2)

## 3. t-SNE sobre digits: efecto de `perplexity`

`perplexity` controla cuántos vecinos efectivos considera cada punto (típico 5-50).

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for a, perp in zip(axes, (5, 30, 50)):
    Z = TSNE(n_components=2, perplexity=perp, init="pca",
             learning_rate="auto", random_state=42).fit_transform(Xd_s)
    a.scatter(Z[:, 0], Z[:, 1], c=yd, cmap="tab10", s=10)
    a.set_title(f"perplexity={perp}"); a.set_xticks([]); a.set_yticks([])
plt.suptitle("t-SNE sobre digits: efecto de perplexity")
plt.tight_layout(); plt.show()
print("t-SNE listo (subset de 500 para que sea rapido)")

## 4. Tiempo de t-SNE (y por qué UMAP sería más rápido)

Medimos el tiempo de t-SNE. UMAP (no instalado) suele ser >3x más rápido y preserva mejor la estructura global.

In [ ]:
import time

t0 = time.perf_counter()
Ztsne = TSNE(n_components=2, perplexity=30, init="pca",
             learning_rate="auto", random_state=42).fit_transform(Xd_s)
t_tsne = time.perf_counter() - t0
print(f"t-SNE sobre 500 digits: {t_tsne:.2f} s")
print("UMAP (umap-learn) haria esto varias veces mas rapido y con transform sobre datos nuevos.")
assert Ztsne.shape == (500, 2)

## 5. LDA supervisado: reducir y clasificar

LDA usa las etiquetas `y` y queda topado en `n_clases - 1` dimensiones (aquí 9). Comparamos accuracy en 2D, 9D y con las 64 features originales.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

def acc(X):
    return cross_val_score(LogisticRegression(max_iter=2000), X, yd_full, cv=5, n_jobs=1).mean()

Xf_s = StandardScaler().fit_transform(Xd_full)
lda2 = LinearDiscriminantAnalysis(n_components=2).fit_transform(Xf_s, yd_full)
lda9 = LinearDiscriminantAnalysis(n_components=9).fit_transform(Xf_s, yd_full)

a2, a9, a64 = acc(lda2), acc(lda9), acc(Xf_s)
print(f"accuracy LDA-2D : {a2:.4f}")
print(f"accuracy LDA-9D : {a9:.4f}")
print(f"accuracy 64D    : {a64:.4f}")
assert a9 >= a2, "9 dimensiones LDA deberian superar a 2"

plt.figure(figsize=(7, 5))
sc = plt.scatter(lda2[:, 0], lda2[:, 1], c=yd_full, cmap="tab10", s=8, alpha=0.6)
plt.colorbar(sc, label="digito")
plt.title("LDA a 2D (supervisado): clases separadas")
plt.xlabel("LD1"); plt.ylabel("LD2")
plt.tight_layout(); plt.show()

## Ejercicios

1. Reducí el swiss roll a 2D con PCA, MDS e Isomap y comentá cuál desenrolla (sección 2).
2. Aplicá t-SNE con `perplexity ∈ {5, 30, 50}` sobre `digits` y explicá el efecto.
3. Medí el tiempo de t-SNE; discutí por qué UMAP sería preferible (velocidad, estructura global, `transform`).
4. Aplicá LDA a 2D y 9D, entrená `LogisticRegression` y compará accuracy con las 64 dimensiones originales.

## Conclusiones

- MDS preserva distancias pairwise; Isomap las geodésicas y por eso desenrolla manifolds curvos.
- t-SNE es excelente para **visualizar** clusters, pero las distancias entre clusters no son fiables y no sirve como preprocesamiento (no tiene `transform` confiable).
- LDA es supervisado: maximiza separación entre clases con pocas dimensiones — útil como preprocesamiento de clasificación.
- UMAP (no instalado acá) suele superar a t-SNE en velocidad y preservación de estructura global.